# MuscleMap WB Segmentation — Augmented Dataset — GPU Optimised

Identical pipeline to `lambda_musclemap_wb_augmented.ipynb` with one key optimisation:

| | Original | Optimised |
|---|---|---|
| Process model | 1 subprocess per volume | 1 subprocess total |
| Model loads | 20 × (one per volume) | 1 (kept in memory) |
| Python startups | 20 | 1 |

A `mm_batch_runner.py` script is written to `/tmp/` and executed once inside the
conda env Python.  It discovers the `mm_segment` CLI callable via
`importlib.metadata` entry-points so no internal API path is hard-coded.

MuscleMap requires Python 3.11; a conda env is created automatically.

Data: `~/our_augmented_dataset/{stem}_augmented000_water.nii.gz`  
Output: `~/musclemap_wb_augmented_segs/{stem}_augmented000_water_dseg.nii.gz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /path/to/our_augmented_dataset/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/our_augmented_dataset/
```

## 2 — Download results
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/musclemap_wb_augmented_segs/ \
  /path/to/local/muscle_map_wb/augmented_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys, os, shutil, glob, platform
import numpy as np
import SimpleITK as sitk

# ── Locate or install conda ───────────────────────────────────────────────────
_conda_candidates = [
    shutil.which('conda'),
    os.path.expanduser('~/miniconda3/bin/conda'),
    os.path.expanduser('~/anaconda3/bin/conda'),
    '/opt/conda/bin/conda',
]
CONDA = next((p for p in _conda_candidates if p and os.path.exists(p)), None)
if CONDA is None:
    _arch = 'aarch64' if platform.machine() == 'aarch64' else 'x86_64'
    _url  = f'https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-{_arch}.sh'
    # -u allows overwriting a partial/broken existing installation
    subprocess.check_call(['bash', '-c',
        f'wget -q {_url} -O /tmp/miniconda.sh && bash /tmp/miniconda.sh -b -u -p ~/miniconda3'])
    CONDA = os.path.expanduser('~/miniconda3/bin/conda')

ENV_NAME = 'musclemap_env'
ENV_DIR  = os.path.join(os.path.dirname(os.path.dirname(CONDA)), 'envs', ENV_NAME)
ENV_PY   = os.path.join(ENV_DIR, 'bin', 'python')
MM_BIN   = os.path.join(ENV_DIR, 'bin', 'mm_segment')

if not os.path.exists(ENV_PY):
    subprocess.check_call([CONDA, 'create', '-n', ENV_NAME, 'python=3.11', 'pip',
                           '-c', 'conda-forge', '--override-channels', '-y', '-q'])
subprocess.check_call([ENV_PY, '-m', 'pip', 'install', '-q',
                       'git+https://github.com/MuscleMap/MuscleMap.git'])
print('mm_segment exists:', os.path.exists(MM_BIN))

In [ ]:
DATA_DIR   = os.path.expanduser('~/our_augmented_dataset')
OUTPUT_DIR = os.path.expanduser('~/musclemap_wb_augmented_segs')

os.makedirs(OUTPUT_DIR, exist_ok=True)

nii_files = sorted(glob.glob(os.path.join(DATA_DIR, '*_augmented*_water.nii.gz')))
print(f'Found {len(nii_files)} NIfTI water volumes')
for p in nii_files:
    print(' ', os.path.basename(p))

In [ ]:
import json

BATCH_RUNNER = '/tmp/mm_batch_runner.py'

_RUNNER_SRC = '''
#!/usr/bin/env python3
"""
MuscleMap batch runner — single process for all volumes.

Optimization over one-subprocess-per-volume:
  - Python interpreter starts ONCE (not N times)
  - All packages imported ONCE
  - If MuscleMap caches the model in module globals, it is loaded ONCE

Strategy order:
  1. importlib.metadata entry_points  — discovers mm_segment callable from
     package metadata; no hard-coded module path required
  2. Direct import of common CLI module names  — fallback if entry_points fails
  3. subprocess per volume  — guaranteed fallback (identical to original)
"""
import sys, os, json, shutil, glob, subprocess, importlib

nii_files  = json.loads(sys.argv[1])
output_dir = sys.argv[2]
use_gpu    = sys.argv[3].lower() == \'true\'

os.makedirs(output_dir, exist_ok=True)

MM_BIN  = os.path.join(os.path.dirname(sys.executable), \'mm_segment\')
GPU_ARG = \'Y\' if use_gpu else \'N\'

run_env = os.environ.copy()
run_env.pop(\'MPLBACKEND\', None)


def _fix_output(nii_path, output_dir):
    """mm_segment may name output differently; rename to our expected convention."""
    stem     = os.path.basename(nii_path).replace(\'.nii.gz\', \'\')  
    expected = os.path.join(output_dir, f\'{stem}_dseg.nii.gz\')
    if os.path.exists(expected):
        return expected
    cands = glob.glob(os.path.join(output_dir, f\'{stem}*dseg*.nii.gz\'))
    if cands:
        shutil.move(cands[0], expected)
    return expected


# ── Strategy 1: entry_points (most robust — no hard-coded module path) ─────────
_cli_fn = None
try:
    from importlib.metadata import entry_points
    eps   = entry_points(group=\'console_scripts\')
    mm_ep = next((ep for ep in eps if ep.name == \'mm_segment\'), None)
    if mm_ep:
        _cli_fn = mm_ep.load()
        print(f\'[runner] Entry point: {mm_ep.value}\', flush=True)
except Exception as e:
    print(f\'[runner] entry_points lookup failed: {e}\', flush=True)

# ── Strategy 2: well-known CLI module paths ─────────────────────────────────
if _cli_fn is None:
    for _path in (\'musclemap.cli:main\', \'musclemap.scripts.mm_segment:main\',
                  \'musclemap.main:main\', \'musclemap.run:main\'):
        try:
            _mod_name, _attr = _path.split(\':\')
            _mod = importlib.import_module(_mod_name)
            _fn  = getattr(_mod, _attr, None)
            if _fn is not None:
                _cli_fn = _fn
                print(f\'[runner] CLI function: {_path}\', flush=True)
                break
        except ImportError:
            continue

if _cli_fn is None:
    print(\'[runner] No CLI function found — subprocess per volume (fallback)\', flush=True)

# ── Main loop ──────────────────────────────────────────────────────────────────
total = len(nii_files)
done  = 0

for i, nii_path in enumerate(nii_files):
    stem     = os.path.basename(nii_path).replace(\'.nii.gz\', \'\')  
    expected = os.path.join(output_dir, f\'{stem}_dseg.nii.gz\')

    if os.path.exists(expected):
        print(f\'[runner] Skipping (done): {stem}\', flush=True)
        done += 1
        continue

    print(f\'\\n[runner] {stem}  ({i+1}/{total})\', flush=True)

    if _cli_fn is not None:
        _saved_argv = sys.argv[:]
        sys.argv = [\'mm_segment\', \'-i\', nii_path, \'-r\', \'wholebody\',
                    \'-o\', output_dir, \'-g\', GPU_ARG]
        try:
            _cli_fn()
        except SystemExit:
            pass
        finally:
            sys.argv = _saved_argv
    else:
        subprocess.check_call(
            [MM_BIN, \'-i\', nii_path, \'-r\', \'wholebody\',
             \'-o\', output_dir, \'-g\', GPU_ARG],
            env=run_env,
        )

    _fix_output(nii_path, output_dir)
    if os.path.exists(expected):
        done += 1
        print(f\'[runner] → {expected}\', flush=True)
    else:
        print(f\'[runner] WARNING: output not found for {stem}\', flush=True)

print(f\'\\n[runner] Finished. {done}/{total} volumes.\', flush=True)
'''

with open(BATCH_RUNNER, 'w') as _f:
    _f.write(_RUNNER_SRC)
print(f'Batch runner written to {BATCH_RUNNER}')

In [ ]:
import json

use_gpu    = True
files_json = json.dumps(nii_files)
run_env    = os.environ.copy()
run_env.pop('MPLBACKEND', None)

# Single subprocess call — the batch runner handles all volumes internally.
subprocess.check_call(
    [ENV_PY, BATCH_RUNNER, files_json, OUTPUT_DIR, str(use_gpu)],
    env=run_env,
)
print('\nBatch runner complete.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*_dseg.nii.gz')))
print(f'Output files: {len(results)} / {len(nii_files)}')
if results:
    arr = sitk.GetArrayFromImage(sitk.ReadImage(results[0]))
    print(f'Sample shape: {arr.shape}  labels: {sorted(np.unique(arr[arr>0]).tolist())[:10]}')